### 2. Data Cleaning

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data/raw"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
nasa = pd.read_csv(
    RAW_DIR / "nasa_power/nasa_power_raw.csv"
)

open_meteo = pd.read_csv(
    RAW_DIR / "open_meteo/open_meteo_raw.csv"
)

osm = pd.read_csv(
    RAW_DIR / "osm/locations_raw.csv"
)

print("NASA POWER:", nasa.shape)
print("Open-Meteo:", open_meteo.shape)
print("OpenStreetMap:", osm.shape)

NASA POWER: (5117, 10)
Open-Meteo: (5117, 11)
OpenStreetMap: (7, 8)


#### Date formats

In [3]:
nasa["date"] = pd.to_datetime(nasa["date"])
open_meteo["date"] = pd.to_datetime(open_meteo["date"])

In [4]:
print(nasa["date"].dtype)
print(open_meteo["date"].dtype)

datetime64[us]
datetime64[us]


#### Remove duplicates

In [5]:
nasa = nasa.drop_duplicates(
    subset=["date", "location"]
).copy()

open_meteo = open_meteo.drop_duplicates(
    subset=["date", "location"]
).copy()

In [6]:
print(
    "NASA duplicate date-location records:",
    nasa.duplicated(["date", "location"]).sum()
)

print(
    "Open-Meteo duplicate date-location records:",
    open_meteo.duplicated(["date", "location"]).sum()
)

NASA duplicate date-location records: 0
Open-Meteo duplicate date-location records: 0


#### Fix the Delhi city value

In [7]:
osm["city"] = osm["city"].fillna(osm["location"])

In [8]:
osm[osm["location"] == "Delhi"]

,location,osm_latitude,osm_longitude,display_name,city,state,country,country_code
0,Delhi,28.632803,77.219771,"Delhi, India",Delhi,Delhi,India,in


In [9]:
osm.isnull().sum()

location         0
osm_latitude     0
osm_longitude    0
display_name     0
city             0
state            0
country          0
country_code     0
dtype: int64

#### Standardize text

In [10]:
for df in [nasa, open_meteo, osm]:
    df["location"] = df["location"].str.strip()

In [11]:
osm["city"] = osm["city"].str.strip()
osm["state"] = osm["state"].str.strip()
osm["country"] = osm["country"].str.strip()

#### Check numeric values

In [12]:
open_meteo[
    [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ]
].describe()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
count,5117.000000,5117.000000,5117.000000,5117.000000,5117.000000
mean,30.742036,20.791049,25.298534,4.336936,15.355677
std,4.812514,5.325723,4.860486,10.381172,5.536613
min,14.100000,3.200000,8.200000,0.000000,4.400000
25%,28.000000,17.500000,22.600000,0.000000,10.900000
50%,30.800000,21.400000,26.100000,0.000000,14.800000
75%,33.500000,25.000000,28.300000,3.300000,18.900000
max,46.600000,34.500000,39.200000,116.000000,44.000000


#### Check logical temperature consistency

In [13]:
invalid_temp = open_meteo[
    open_meteo["temperature_2m_min"] >
    open_meteo["temperature_2m_max"]
]

print("Invalid temperature records:", len(invalid_temp))

Invalid temperature records: 0


In [14]:
invalid_mean = open_meteo[
    (open_meteo["temperature_2m_mean"] >
     open_meteo["temperature_2m_max"])
    |
    (open_meteo["temperature_2m_mean"] <
     open_meteo["temperature_2m_min"])
]

print("Invalid mean-temperature records:", len(invalid_mean))

Invalid mean-temperature records: 0


#### Check missing values after cleaning

In [15]:
print("NASA POWER missing values:")
print(nasa.isnull().sum())

print("\nOpen-Meteo missing values:")
print(open_meteo.isnull().sum())

print("\nOpenStreetMap missing values:")
print(osm.isnull().sum())

NASA POWER missing values:
date                 0
T2M                  0
RH2M                 0
WS2M                 0
ALLSKY_SFC_SW_DWN    0
location             0
latitude             0
longitude            0
state                0
region               0
dtype: int64

Open-Meteo missing values:
date                   0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
wind_speed_10m_max     0
location               0
latitude               0
longitude              0
state                  0
region                 0
dtype: int64

OpenStreetMap missing values:
location         0
osm_latitude     0
osm_longitude    0
display_name     0
city             0
state            0
country          0
country_code     0
dtype: int64


#### Save cleaned datasets

In [16]:
nasa.to_csv(
    PROCESSED_DIR / "nasa_power_clean.csv",
    index=False
)

open_meteo.to_csv(
    PROCESSED_DIR / "open_meteo_clean.csv",
    index=False
)

osm.to_csv(
    PROCESSED_DIR / "locations_clean.csv",
    index=False
)

### Cleaned datasets saved successfully